# DPS Departures — Preprocessing & Agregasi Harian

**Input:** `dps_departures_YYYY-MM.csv` (2025-03 s/d 2026-03)  
**Output:** `dps_departures_daily.csv` = satu baris per hari berisi jumlah early, ontime, delay, cancelled  

**Aturan kategorisasi (IATA standard):**
- `early`     : berangkat > 1 menit lebih awal
- `ontime`    : delay antara -1 s/d +15 menit
- `delay`     : delay > 15 menit
- `cancelled` : status Cancelled atau Diverted

`pct_ontime` dan `pct_delay` dihitung dari **operated flights** (exclude cancelled+diverted).

In [13]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## 1. Load Semua File CSV

In [14]:
DATA_DIR = Path('.')
OUTPUT_FILE = 'dps_departures_daily.csv'
ONTIME_THRESHOLD_MIN = 15     # IATA standard

months = pd.period_range('2025-03', '2026-03', freq='M')
file_names = [f'dps_departures_{m}.csv' for m in months]

In [15]:
dfs = []
missing = []

for f in file_names:
    path = DATA_DIR / f
    if path.exists():
        df_tmp = pd.read_csv(path, dtype={'tanggal': str})
        df_tmp['_source_file'] = f
        dfs.append(df_tmp)
    else:
        missing.append(f)

if missing:
    print(f'{len(missing)} file tidak ditemukan, dilewati: {missing}')

df_raw = pd.concat(dfs, ignore_index=True)
print(f'\nTotal baris gabungan: {len(df_raw):,}')
print(f'Rentang tanggal    : {df_raw["tanggal"].min()} s/d {df_raw["tanggal"].max()}')
print(f'\nDistribusi status:')
print(df_raw['status'].value_counts())


Total baris gabungan: 80,961
Rentang tanggal    : 2025-03-01 s/d 2026-03-31

Distribusi status:
status
Landed       76496
Cancelled     1295
Diverted        29
Name: count, dtype: int64


## 2. Parse Kolom `delay_berangkat` → Menit (Numerik)

In [16]:
def parse_delay(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip().lower()
    if val in ('on time', 'ontime', '0', ''):
        return 0.0
    m = re.match(r'(\d+)\s*min\s*(late|early)', val)
    if m:
        menit = int(m.group(1))
        return menit if m.group(2) == 'late' else -menit

    # fallback
    try:
        return float(val)
    except ValueError:
        return np.nan

df_raw['delay_menit'] = df_raw['delay_berangkat'].apply(parse_delay)


n_unparsed = df_raw['delay_menit'].isna().sum()
n_cancelled = df_raw['status'].isin(['Cancelled', 'Diverted']).sum()
n_null_operated = df_raw[
    df_raw['delay_menit'].isna() & ~df_raw['status'].isin(['Cancelled', 'Diverted'])
].shape[0]

print(f'Total null delay_menit : {n_unparsed:,}')
print(f'  → dari Cancelled/Diverted : {n_cancelled:,}  (wajar)')
print(f'  → dari Landed/operated    : {n_null_operated:,}  (akan di-impute)')

Total null delay_menit : 4,835
  → dari Cancelled/Diverted : 1,324  (wajar)
  → dari Landed/operated    : 3,511  (akan di-impute)


## 3. Impute Null pada Operated Flights

Flight dengan status `Landed` tapi `delay_berangkat` kosong:
semua kasus menunjukkan `jadwal_lokal == berangkat_aktual`,
artinya **tepat waktu → di-impute 0 menit**.

In [17]:
mask_operated_null = (
    df_raw['delay_menit'].isna() &
    ~df_raw['status'].isin(['Cancelled', 'Diverted'])
)

operated_null = df_raw[mask_operated_null][[
    'tanggal', 'flight_iata', 'airline',
    'jadwal_lokal', 'berangkat_aktual', 'delay_berangkat', 'status'
]]
print(f'Operated flights dengan null delay: {len(operated_null)}')
display(operated_null.head(10))

same_time = (
    df_raw.loc[mask_operated_null, 'jadwal_lokal'] ==
    df_raw.loc[mask_operated_null, 'berangkat_aktual']
).sum()
print(f'\nDari {mask_operated_null.sum()} baris null: {same_time} punya jadwal == berangkat_aktual → impute 0')

df_raw.loc[mask_operated_null, 'delay_menit'] = 0.0
print('Impute selesai')

Operated flights dengan null delay: 3511


,tanggal,flight_iata,airline,jadwal_lokal,berangkat_aktual,delay_berangkat,status
22,2025-03-01,JT856,Lion Air,07:00,NaN,NaN,NaN
67,2025-03-01,0B772,Blue Air,12:50,NaN,NaN,NaN
127,2025-03-01,IN281,NAM Air,17:45,NaN,NaN,NaN
147,2025-03-01,IU727,SW Italia,20:55,NaN,NaN,NaN
151,2025-03-01,IP105,Pelita Air Service,21:15,NaN,NaN,NaN
185,2025-03-02,JT856,Lion Air,07:00,NaN,NaN,NaN
220,2025-03-03,SJ726,Sriwijaya Air,01:00,NaN,NaN,NaN
236,2025-03-03,8B5101,TransNusa,10:30,NaN,NaN,NaN
259,2025-03-03,0B772,Blue Air,12:50,NaN,NaN,NaN
287,2025-03-03,8B5103,TransNusa,15:00,NaN,NaN,NaN



Dari 3511 baris null: 370 punya jadwal == berangkat_aktual → impute 0
Impute selesai


## 4. Kategorisasi Per-Flight

In [18]:
def kategorisasi(row):
    if row['status'] in ('Cancelled', 'Diverted'):
        return 'cancelled'
    d = row['delay_menit']
    if pd.isna(d):
        return 'unknown'
    if d < -1:
        return 'early'
    elif d <= ONTIME_THRESHOLD_MIN:
        return 'ontime'
    else:
        return 'delay'

df_raw['kategori'] = df_raw.apply(kategorisasi, axis=1)

print('Distribusi kategori (semua bulan):')
print(df_raw['kategori'].value_counts())
print(f'\nTotal unknown (harusnya 0): {(df_raw["kategori"] == "unknown").sum()}')

Distribusi kategori (semua bulan):
kategori
delay        38177
ontime       34487
early         6973
cancelled     1324
Name: count, dtype: int64

Total unknown (harusnya 0): 0


## 5. Agregasi Per Hari

In [19]:
df_raw['tanggal'] = pd.to_datetime(df_raw['tanggal'])

agg_counts = (
    df_raw.groupby(['tanggal', 'kategori'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ['early', 'ontime', 'delay', 'cancelled', 'unknown']:
    if col not in agg_counts.columns:
        agg_counts[col] = 0

avg_delay = (
    df_raw[df_raw['kategori'] == 'delay']
    .groupby('tanggal')['delay_menit']
    .mean()
    .reset_index()
    .rename(columns={'delay_menit': 'avg_delay_menit'})
)

daily = agg_counts.merge(avg_delay, on='tanggal', how='left')
daily['total_flight'] = daily['early'] + daily['ontime'] + daily['delay'] + daily['cancelled'] + daily['unknown']
daily['operated']     = daily['early'] + daily['ontime'] + daily['delay'] + daily['unknown']
daily['pct_ontime']   = np.where(
    daily['operated'] > 0,
    (daily['early'] + daily['ontime']) / daily['operated'] * 100,
    np.nan
)
daily['pct_delay']    = np.where(
    daily['operated'] > 0,
    daily['delay'] / daily['operated'] * 100,
    np.nan
)
daily['avg_delay_menit'] = daily['avg_delay_menit'].round(1)
daily['pct_ontime']      = daily['pct_ontime'].round(1)
daily['pct_delay']       = daily['pct_delay'].round(1)

daily = daily.drop(columns=['unknown'], errors='ignore')

daily = daily[[
    'tanggal', 'total_flight', 'early', 'ontime', 'delay',
    'cancelled', 'operated', 'pct_ontime', 'pct_delay', 'avg_delay_menit'
]].sort_values('tanggal').reset_index(drop=True)

print(f'Jumlah baris harian: {len(daily)}')
print(f'Rentang: {daily["tanggal"].min().date()} s/d {daily["tanggal"].max().date()}')
display(daily.head(10))

Jumlah baris harian: 396
Rentang: 2025-03-01 s/d 2026-03-31


,tanggal,total_flight,early,ontime,delay,cancelled,operated,pct_ontime,pct_delay,avg_delay_menit
0,2025-03-01,170,36,80,44,10,160,72.50,27.50,27.90
1,2025-03-02,43,13,24,5,1,42,88.10,11.90,28.60
2,2025-03-03,150,35,74,37,4,146,74.70,25.30,31.50
3,2025-03-04,160,39,90,28,3,157,82.20,17.80,26.40
4,2025-03-05,171,28,109,33,1,170,80.60,19.40,26.70
5,2025-03-06,173,46,82,27,18,155,82.60,17.40,30.20
6,2025-03-07,186,42,98,31,15,171,81.90,18.10,33.70
7,2025-03-08,173,17,63,79,14,159,50.30,49.70,37.50
8,2025-03-09,189,34,91,53,11,178,70.20,29.80,29.50
9,2025-03-10,176,27,95,46,8,168,72.60,27.40,31.00


## 6. Validasi & Ringkasan

In [20]:
total_daily = daily['total_flight'].sum()
total_raw   = len(df_raw)
print(f'Total flight (raw)   : {total_raw:,}')
print(f'Total flight (daily) : {total_daily:,}')
print(f'Match                : {"Benar" if total_daily == total_raw else "mismatch"}')

print()
print('Null per kolom:')
print(daily.isna().sum())

print()
print('Statistik ringkas:')
display(daily[['total_flight','early','ontime','delay','cancelled','pct_ontime','pct_delay','avg_delay_menit']].describe().round(2))

Total flight (raw)   : 80,961
Total flight (daily) : 80,961
Match                : Benar

Null per kolom:
tanggal            0
total_flight       0
early              0
ontime             0
delay              0
cancelled          0
operated           0
pct_ontime         0
pct_delay          0
avg_delay_menit    0
dtype: int64

Statistik ringkas:


,total_flight,early,ontime,delay,cancelled,pct_ontime,pct_delay,avg_delay_menit
count,396.00,396.00,396.00,396.00,396.00,396.00,396.00,396.00
mean,204.45,17.61,87.09,96.41,3.34,52.50,47.50,31.32
std,24.53,7.37,14.97,23.32,4.84,8.74,8.74,1.80
min,43.00,3.00,24.00,5.00,0.00,32.70,11.90,24.80
25%,199.00,12.00,78.00,84.00,0.00,46.70,42.62,30.10
50%,206.00,17.00,86.00,98.00,1.00,52.00,48.00,31.30
75%,211.00,22.00,95.00,110.00,4.00,57.38,53.30,32.50
max,517.00,46.00,221.00,259.00,28.00,88.10,67.30,37.50


In [21]:
print('10 hari dengan on-time rate terendah:')
display(
    daily.nsmallest(10, 'pct_ontime')[[
        'tanggal', 'total_flight', 'early', 'ontime',
        'delay', 'cancelled', 'pct_ontime', 'avg_delay_menit'
    ]]
)

10 hari dengan on-time rate terendah:


,tanggal,total_flight,early,ontime,delay,cancelled,pct_ontime,avg_delay_menit
116,2025-06-25,209,13,55,140,1,32.70,36.20
111,2025-06-20,209,11,58,137,3,33.50,34.80
119,2025-06-28,205,10,60,135,0,34.10,34.10
146,2025-07-25,215,6,67,140,2,34.30,32.30
110,2025-06-19,205,8,64,129,4,35.80,32.50
125,2025-07-04,219,8,71,140,0,36.10,31.80
382,2026-03-18,215,7,71,136,1,36.40,29.60
114,2025-06-23,204,15,59,128,2,36.60,34.00
117,2025-06-26,207,10,66,130,1,36.90,34.80
294,2025-12-20,207,9,68,130,0,37.20,33.20


## 7. Export ke CSV

In [22]:
daily.to_csv(OUTPUT_FILE, index=False, date_format='%Y-%m-%d')
print(f'File tersimpan: {OUTPUT_FILE}')
print(f'Baris  : {len(daily):,}')
print(f'Kolom  : {list(daily.columns)}')

File tersimpan: dps_departures_daily.csv
Baris  : 396
Kolom  : ['tanggal', 'total_flight', 'early', 'ontime', 'delay', 'cancelled', 'operated', 'pct_ontime', 'pct_delay', 'avg_delay_menit']
